In [25]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, train_test_split

In [26]:
seed = 1234
np.random.seed(seed)

## Load Data

In [27]:
data = pd.read_csv('./data/training.csv')
test_data = pd.read_csv('./data/public_processed.csv')

In [28]:
group_map = pd.read_csv('./preprocess_data/output.txt', header=None, delimiter=' ')
group_map_dict = group_map.set_index(0)[1].to_dict()

## Data Analysis

In [29]:
data.isnull().sum()

txkey              0
locdt              0
loctm              0
chid               0
cano               0
contp              0
etymd         203455
mchno              0
acqic              0
mcc             4550
conam              0
ecfg               0
insfg              0
iterm              0
bnsfg              0
flam1              0
stocn            600
scity         266066
stscd        8665195
ovrlt              0
flbmk              0
hcefg         286656
csmcu         498657
csmam              0
flg_3dsmk          0
label              0
dtype: int64

In [30]:
test_data.isnull().sum()

txkey             0
locdt             0
loctm             0
chid              0
cano              0
contp             0
etymd         13826
mchno             0
acqic             0
mcc             354
conam             0
ecfg              0
insfg             0
iterm             0
bnsfg             0
flam1             0
stocn            70
scity         18430
stscd        598703
ovrlt             0
flbmk             0
hcefg         19703
csmcu         33882
csmam             0
flg_3dsmk         0
dtype: int64

In [31]:
num_feat = [
    'locdt',
    'loctm',
    'conam',
    'iterm',
    'flam1',
    'csmam',
]
cat_feat = [col for col in data.columns if col not in num_feat]

In [32]:
print('Unique Count:')
for feat in cat_feat:
    count = data[feat].nunique()
    print(f'- {feat}: {count}')

Unique Count:
- txkey: 8688526
- chid: 482667
- cano: 618898
- contp: 7
- etymd: 10
- mchno: 163797
- acqic: 8334
- mcc: 459
- ecfg: 2
- insfg: 2
- bnsfg: 2
- stocn: 122
- scity: 12003
- stscd: 5
- ovrlt: 2
- flbmk: 2
- hcefg: 11
- csmcu: 79
- flg_3dsmk: 2
- label: 2


## Data Preprocessing
### Steps
1. ~~Fill Missing Data~~ Remove Samples with Missing Data
    - missed data are all categorical data
    - perhaps N/A should be seen as a category
2. Encoding
    - categorical -> one-hot
        - feature with numerous categories is not suitable for one-hot encoding
    - numerical -> normalization (0 ~ 1)
3. Data Balancing
4. Feature Extraction
    1. column removal by intuition
    2. column removal with PCA
    3. column removal by intuition + PCA


In [33]:
# by intuition
removed_feat = [
    'txkey',  # 交易序號
    'cano',   # 卡號
    'mchno',  # 特店代號
    'acqic',  # 收單行代碼
    'csmam',  # 消費地金額
]

data = data.drop(columns=removed_feat)
data = data.dropna()  # drop rows with N/A value
data['chid'] = data['chid'].map(group_map_dict)

In [34]:
test_txkey = test_data['txkey']
test_data = test_data.drop(columns=removed_feat)
test_data['chid'] = test_data['chid'].map(group_map_dict)

In [35]:
labels = data['label']
data = data.drop(columns='label')
X_train, X_test, y_train, y_test = train_test_split(data, labels, test_size=0.15)

## Model Construction
- random forest
- SVM
- Logistic Regression
- ~~XGBoost~~ LightGBM

In [36]:
model = lgb.LGBMClassifier(
    metric='l1',
    num_leaves=38,
)

## Training & Validation
- 800-fold cross validation

In [37]:
model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 8900, number of negative: 9562
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000902 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1642
[LightGBM] [Info] Number of data points in the train set: 18462, number of used features: 17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.482071 -> initscore=-0.071746
[LightGBM] [Info] Start training from score -0.071746


LGBMClassifier(metric='l1', num_leaves=38)

## Prediction

In [39]:
y_pred = model.predict(test_data)
res = {
    'txkey': test_txkey,
    'label': y_pred
}

pd.DataFrame(res).to_csv('pred.csv', index=False)